# MNIST-Planted-Cov: the dataset, nothing else

Just generates and looks at the dataset produced by `configs/data/mnist_planted_cov.yaml`
with this run's overrides -- no model, no training:

    experiment:                   planted_parents
    kappa:                        0.9
    num_covariates:               5
    corruption:                   gaussian
    corruption_strength:          0.4   (channel 5 / X only)
    concept_corruption_strength:  0.0   (channels 1-4, clean)

`features` is a `[num_covariates, 28, 28]` tensor, one MNIST digit image per channel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf

from datasets.MNIST_add_cov_dataset import (
    get_MNIST_add_cov_datasets,
    get_mnist_add_cov_concept_names,
    get_mnist_add_cov_oracle_names,
    summarize_dataset,
)

config_data = OmegaConf.merge(
    OmegaConf.load("configs/data/data_defaults.yaml"),
    OmegaConf.load("configs/data/mnist_planted_cov.yaml"),
)
config_data.data_path = "datasets/"   # local path; the cluster uses work/Data/

# This run's overrides
config_data.experiment = "planted_parents"
config_data.kappa = 0.9
config_data.num_covariates = 5
config_data.corruption = "gaussian"
config_data.corruption_strength = 0.4
config_data.concept_corruption_strength = 0.0

print(OmegaConf.to_yaml(config_data))

## Generate

In [ ]:
trainset, valset, testset = get_MNIST_add_cov_datasets(config_data, seed=0)
concept_names = get_mnist_add_cov_concept_names(config_data.experiment)
oracle_names = get_mnist_add_cov_oracle_names(config_data.experiment)
print(len(trainset), len(valset), len(testset))
print(concept_names, "| oracle:", oracle_names)

## Summary

Marginal concept-X correlations, the concept-only task ceiling the residual has to beat,
and whether X is determined by (A, B) -- the sanity check baked into the dataset script
itself (`summarize_dataset`).

In [ ]:
summarize_dataset(trainset)

## One sample

In [ ]:
def show_sample(dataset, index, axes=None):
    """Plot every digit channel of one sample, titled with its digits and labels."""
    item = dataset[index]
    features = item["features"]                       # [num_covariates, 28, 28]
    concepts = item["concepts"].tolist()
    hidden = item["hidden_concepts"].tolist()
    digits = item["digit_labels"].tolist()
    short_names = [n.split("::")[0] for n in concept_names]
    short_oracle = [n.split("::")[0] for n in oracle_names]

    n_ch = features.shape[0]
    if axes is None:
        _, axes = plt.subplots(n_ch, 1, figsize=(2.0, 2.3 * n_ch))
    for channel, ax in enumerate(axes):
        ax.imshow(features[channel], cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
    axes[0].set_title("d = " + " ".join(str(d) for d in digits), fontsize=9)
    axes[-1].set_xlabel(
        " ".join(f"{n}={v:.0f}" for n, v in zip(short_names, concepts))
        + "\n[" + " ".join(f"{n}={v:.0f}" for n, v in zip(short_oracle, hidden))
        + f"]  y={int(item['labels'])}",
        fontsize=8,
    )
    return axes


show_sample(trainset, 0)
plt.tight_layout()
plt.show()

## A grid of samples

In [ ]:
def show_samples(dataset, indices, suptitle=""):
    indices = list(indices)
    n_ch = dataset[indices[0]]["features"].shape[0]
    fig, axes = plt.subplots(n_ch, len(indices), figsize=(2.0 * len(indices), 2.3 * n_ch))
    for column, index in enumerate(indices):
        show_sample(dataset, index, axes=axes[:, column])
    for ch in range(n_ch):
        axes[ch, 0].set_ylabel(f"channel {ch}")
    fig.suptitle(
        suptitle or f"MNIST-Planted-Cov ({config_data.experiment}, kappa={config_data.kappa}): "
        f"features[{n_ch}, 28, 28]"
    )
    fig.tight_layout()
    return fig


show_samples(trainset, range(8))
plt.show()

## Class / concept balance

In [ ]:
task_labels = trainset.task_labels
concept_means = trainset.observed_concepts.mean(0)
x_mean = trainset.hidden_concepts[:, 1].mean()

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

axes[0].bar(range(trainset.num_classes), np.bincount(task_labels, minlength=trainset.num_classes))
axes[0].set_xlabel("task label y")
axes[0].set_ylabel("count")
axes[0].set_title("Task class balance (train)")

names = [n.split("::")[0] for n in trainset.observed_concept_names] + ["X"]
means = list(concept_means) + [x_mean]
axes[1].bar(names, means)
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1)
axes[1].set_ylim(0, 1)
axes[1].set_title("Concept / hidden-X means (train)")

fig.tight_layout()
plt.show()